# F1-scientific-python — Session 02: Broadcasting, Vectorization, and Axis Aggregations

**Session length:** about 90 minutes • **Concepts:** elementwise-ops,
broadcasting, vectorization, aggregation-axis

This is the heart of the unit: doing *all* your math as whole-array operations.
By the end you should be able to take loop code and mechanically rewrite it
loop-free — a skill some USAAIO exam tasks require outright. Checkpoint answers
are collected at the end of the notebook.

In [ ]:
import numpy as np

## 1. Elementwise operations

Arithmetic on arrays happens **element by element**: `a + b` adds matching
elements, `a * b` multiplies them, and so on for `-`, `/`, and `**`. An
operation between an array and a single number applies that number to *every*
element. Comparisons (`>`, `<=`, `==`, ...) are elementwise too — that is where
Session 01's boolean masks came from.

NumPy also ships elementwise versions of the standard math functions: `np.sqrt`,
`np.exp`, `np.log`, `np.abs`, `np.sin`, `np.round`, and many more. Each takes an
array and returns a new array of the same shape with the function applied to
every element.

In [ ]:
a = np.array([1.0, 4.0, 9.0, 16.0])
b = np.array([10.0, 20.0, 30.0, 40.0])

print("a + b:      ", a + b)
print("b / a:      ", b / a)
print("a ** 2:     ", a ** 2)
print("a + 100:    ", a + 100)          # number applied to every element
print("np.sqrt(a): ", np.sqrt(a))
print("a > 5:      ", a > 5)

A worked example — converting a week of Celsius readings to Fahrenheit with the
formula $F = \frac{9}{5}C + 32$, applied to all readings at once:

In [ ]:
celsius = np.array([-5.0, 0.0, 12.5, 21.0, 30.0])
fahrenheit = celsius * 9 / 5 + 32
print(fahrenheit)

### Checkpoint 1

1. Given `radii = np.array([1.0, 2.0, 3.5])`, compute the area of each circle
   ($\pi r^2$) in one line. (`np.pi` holds $\pi$.)
2. Predict the output of `np.array([2, 4, 6]) ** np.array([1, 2, 3])`, then
   check.

## 2. The elementwise toolkit: np.where and np.maximum

Two more tools you will use constantly:

- `np.maximum(a, b)` compares two arrays **elementwise** and keeps the larger of
  each pair (`np.minimum` keeps the smaller). Careful: this is different from
  `a.max()`, which collapses one array to its single largest element.
- `np.where(condition, x, y)` builds a new array that takes from `x` where the
  condition is True and from `y` where it is False — an elementwise if/else.

In [ ]:
a = np.array([3, 8, 1, 9])
b = np.array([5, 2, 7, 4])
print("np.maximum(a, b):", np.maximum(a, b))
print("a.max():         ", a.max())

scores = np.array([48, 72, 95, 60, 33])
labels = np.where(scores >= 60, "pass", "retry")     # elementwise if/else
print("labels:          ", labels)
print("floor at 50:     ", np.where(scores < 50, 50, scores))

### Checkpoint 2

1. Use `np.where` to replace every value bigger than 10 in `np.arange(15)`
   with 10.
2. In one sentence: what is the difference between `np.maximum(a, b)` and
   `a.max()`?

## 3. Broadcasting: the shape-compatibility rules

Elementwise math needs two arrays of matching shape — one element from each.
But you have already seen an exception work: `a + 100` stretched the single
number `100` across every element. **Broadcasting** is the general rule for
when NumPy will stretch shapes like that, and it is one of the most powerful
ideas in this unit.

To combine two arrays, NumPy compares their shapes **from the right end**, one
axis at a time:

1. If one shape has fewer axes, imagine extra axes of size 1 on its **left**.
2. Two axis sizes are compatible if they are **equal**, or if **one of them
   is 1**.
3. Every size-1 axis is stretched (its values repeated) to match the other
   size. If any pair of sizes is unequal and neither is 1, the operation fails.

The result's shape takes the larger size at each position. Some shape pairs
checked by hand:

```text
(2, 3)  and  (3,)    ->  (3,) is padded to (1, 3); 1 stretches to 2  ->  result (2, 3)
(3, 1)  and  (1, 4)  ->  3 vs 1: stretch; 1 vs 4: stretch            ->  result (3, 4)
(4, 5)  and  (4,)    ->  (4,) pads to (1, 4); 5 vs 4 mismatch        ->  ERROR
```

In [ ]:
g = np.arange(12).reshape(3, 4)
row_bonus = np.array([10, 20, 30, 40])       # shape (4,)

print(g + row_bonus)          # (3, 4) + (4,): the row of bonuses is added to EVERY row

And when shapes are truly incompatible, NumPy refuses loudly rather than
guessing:

In [ ]:
try:
    np.ones(3) + np.ones(4)      # 3 vs 4: unequal, neither is 1
except ValueError as err:
    print("ValueError:", err)

### Checkpoint 3

1. Predict the result shape (or ERROR) for each pair, then verify with
   `np.ones`: `(5, 1) + (1, 6)`; `(2, 3) + (3,)`; `(3,) + (4,)`.
2. In your own words: why is `(4, 5)` with `(4,)` an error even though a 4
   appears in both shapes?

## 4. Building grids with size-1 axes

The starred pattern: shape `(3, 1)` combined with shape `(1, 4)`. Each
stretches along its size-1 axis, producing every pairing of the two — a full
`(3, 4)` grid from 3 + 4 numbers. To make the shapes, use `reshape` or index
with `None`, which inserts a new size-1 axis right where you write it.

In [ ]:
left = np.array([1, 2, 3]).reshape(3, 1)     # shape (3, 1): a column
top = np.array([10, 20, 30, 40])             # shape (4,)
top_row = top[None, :]                        # shape (1, 4): a row (None adds an axis)

print("left * top_row  ->  shape", (left * top_row).shape)
print(left * top_row)

# The same trick builds a times table in one line:
n = np.arange(1, 6)
print(n[:, None] * n[None, :])               # (5, 1) * (1, 5) -> (5, 5)

### Checkpoint 4

1. Build the 10×10 times table for 1 through 10 using one line of broadcasting.
2. With `x = np.arange(4)`, predict the shape and the value at position
   `[2, 1]` of `x[:, None] - x[None, :]`, then check.

## 5. Vectorization: replacing loops with array operations

**Vectorization** means expressing a computation as whole-array operations
instead of an explicit Python loop. You have been doing it all session; now we
make it a deliberate skill: take loop code, produce loop-free code that
computes exactly the same thing.

Why this matters beyond speed: **some USAAIO exam tasks ban Python loops
outright.** A task statement may say that your solution must not contain `for`
or `while`, and submissions are checked for it. On those tasks, writing
loop-free NumPy is not a style preference — it is the difference between a
scoring solution and a zero. Practice until the translation is automatic.

The core translation table:

| Loop pattern | Array replacement |
| --- | --- |
| transform every element | elementwise math / `np.sqrt`-style functions |
| `if`/`else` inside the loop | `np.where(cond, x, y)` or mask assignment |
| running total / count | `.sum()` — and `mask.sum()` counts True values |
| track the biggest/smallest | `.max()` / `.min()` |
| nested loop over two arrays | broadcasting with `[:, None]` and `[None, :]` |

Worked example 1 — total of squares of the positive values:

In [ ]:
values = np.array([3.0, -1.0, 4.0, -1.0, 5.0, -9.0, 2.0])

# Loop version
total_loop = 0.0
for v in values:
    if v > 0:
        total_loop += v ** 2

# Vectorized version: mask, square, sum — no loop
total_vec = (values[values > 0] ** 2).sum()

print(total_loop, total_vec)

Worked example 2 — an if/else per element. Tickets cost 12, but ages under 13
pay 7:

In [ ]:
ages = np.array([5, 21, 12, 40, 13, 8])

# Loop version
prices_loop = []
for age in ages:
    if age < 13:
        prices_loop.append(7)
    else:
        prices_loop.append(12)
prices_loop = np.array(prices_loop)

# Vectorized version
prices_vec = np.where(ages < 13, 7, 12)

print(prices_loop)
print(prices_vec)

Worked example 3 — a **nested** loop building a 2-D table becomes one broadcast
expression:

In [ ]:
a = np.array([2.0, 9.0, 4.0])
b = np.array([1.0, 3.0, 8.0, 6.0])

# Loop version: every (i, j) pairing
diff_loop = np.zeros((3, 4))
for i in range(3):
    for j in range(4):
        diff_loop[i, j] = abs(a[i] - b[j])

# Vectorized: (3, 1) with (1, 4) broadcasts to (3, 4)
diff_vec = np.abs(a[:, None] - b[None, :])

print("identical?", np.array_equal(diff_loop, diff_vec))
print(diff_vec)

When you rewrite a loop, always check your work the same way: run both versions
and compare with `np.array_equal` (exact) or `np.allclose` (allows tiny decimal
rounding differences).

### Checkpoint 5

1. Rewrite this loop-free, in one or two lines:
   ```python
   result = []
   for x in values:
       if x < 0:
           result.append(0)
       else:
           result.append(x)
   total = sum(result)
   ```
2. Loop-free: count how many values in an integer array are even.
   (Hint: `%` works elementwise.)

## 6. Worked exam-style example: a constrained coding task

Here is the register Round 1 actually uses — an exact function name, a shape
contract, and an API ban with a zero-points clause:

> Implement exactly `def total_shipping(weights):`. `weights` is a 1-D array of
> parcel weights in kg. Each parcel costs a flat 5.00 up to 2 kg; heavier
> parcels add 2.00 per kg above 2 kg. Return the **total** cost of all parcels
> as a single number. **Your function body must not contain `for` or `while`;
> submissions containing them score zero for this task.**

Solve it step by step:

1. *Per-parcel if/else* → an elementwise tool. Either `np.where(w <= 2, ...)`
   or, slicker: the surcharge is `2 * (w - 2)` but never below zero, which is
   exactly `2 * np.maximum(w - 2, 0)`.
2. *Total over parcels* → `.sum()`.
3. *Check against a loop* before trusting it (loops are banned in the
   submission, not in your scratch work).

In [ ]:
def total_shipping(weights):
    return (5.0 + 2.0 * np.maximum(weights - 2.0, 0.0)).sum()

# verify against a plain-Python reference on a small test case
w = np.array([0.5, 2.0, 3.5, 6.0])
loop_total = 0.0
for x in w:
    loop_total += 5.0 if x <= 2.0 else 5.0 + 2.0 * (x - 2.0)

print(total_shipping(w), loop_total)

### Checkpoint 6

1. In the same register, implement `def clip_total(values, hi):` — return the
   total of `values` after replacing everything above `hi` with `hi`; no `for`
   or `while`.
2. In one sentence: given the zero-points clause, what should you do to your
   solution *before* submitting it?

## 7. Aggregations along an axis

An **aggregation** collapses many numbers into one summary: `sum`, `mean`,
`max`, `min`. Called plainly — `g.sum()` — it collapses the *entire* array to a
single number.

On a 2-D array, the interesting version aggregates along one **axis**:

- `axis=0` collapses **down the rows**, leaving one summary **per column**;
- `axis=1` collapses **across the columns**, leaving one summary **per row**.

A reliable way to keep this straight: *the axis you pass is the axis that
disappears*. A `(3, 4)` array aggregated with `axis=0` loses the 3, leaving
shape `(4,)`; with `axis=1` it loses the 4, leaving `(3,)`.

In [ ]:
scores = np.array([[80, 92, 70, 88],
                   [55, 64, 90, 71],
                   [98, 73, 85, 60]])     # 3 students x 4 quizzes

print("everything:      ", scores.sum())
print("per quiz (axis=0):   mean =", scores.mean(axis=0))   # shape (4,)
print("per student (axis=1): max =", scores.max(axis=1))    # shape (3,)
print("per student total:       ", scores.sum(axis=1))

`argmax` and `argmin` are aggregations that return the *position* of the
largest/smallest value instead of the value itself — handy for questions like
"*which* quiz had the highest average?".

In [ ]:
print("best quiz by average: index", scores.mean(axis=0).argmax())
print("weakest student by total: index", scores.sum(axis=1).argmin())

### Checkpoint 7

Using `s = np.arange(12).reshape(3, 4)`:

1. Compute the per-column max and the per-row sum. Predict both shapes first.
2. Which row of `s` has the largest total? Answer with one expression.

## 8. keepdims and the row/column recipes

Normally `scores.sum(axis=1)` returns shape `(3,)`. With **`keepdims=True`** it
returns shape `(3, 1)` — the collapsed axis is kept as size 1. Why bother?
Because a `(3, 1)` result **broadcasts** cleanly back against the original
`(3, 4)` array. That is exactly what you need to, say, turn each row into
shares of its row total:

In [ ]:
scores = np.array([[80, 92, 70, 88],
                   [55, 64, 90, 71],
                   [98, 73, 85, 60]])

row_totals = scores.sum(axis=1, keepdims=True)    # shape (3, 1), not (3,)
print("row totals (kept 2-D):")
print(row_totals)

share = scores / row_totals                       # (3, 4) / (3, 1) broadcasts
print("each score as a share of its row total:")
print(share.round(3))
print("rows sum to:", share.sum(axis=1))

Without `keepdims`, dividing `(3, 4)` by `(3,)` would line the 3 up against the
4 (shapes are compared from the right!) and fail. `keepdims` makes the
aggregate line up with the axis it came from.

### Checkpoint 8

Using `s = np.arange(12).reshape(3, 4)`:

1. Compute each element's share of its row total using `keepdims=True`.
2. Why would `s / s.sum(axis=1)` fail, while `s / s.sum(axis=0)` works?

## 9. Common pitfalls II

**Pitfall — the silent shape bug.** Suppose you want to subtract each row's
average from a grid. Aggregating without `keepdims` gives a 1-D result, and
broadcasting lines it up **from the right** — against the *columns*. On a
non-square grid that fails loudly (annoying, but safe). On a **square** grid it
runs without complaint and quietly computes the wrong thing:

In [ ]:
g = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0],
              [10.0, 20.0, 30.0]])       # 3x3: square, so nothing errors

wrong = g - g.mean(axis=1)               # BROKEN: (3,3) minus (3,) pairs with COLUMNS
print("row averages now:", wrong.mean(axis=1).round(3), " <- should be all 0, is not")

fixed = g - g.mean(axis=1, keepdims=True)   # FIX: (3,1) pairs with ROWS
print("row averages now:", fixed.mean(axis=1).round(3))

The habit that catches this: **predict the result's shape before you run, and
check a known property after** (here: "row averages should now be 0").

**Pitfall — axis confusion.** "Average per student" from a
`(students, quizzes)` grid needs the *quiz* axis to disappear — `axis=1`.
Reaching for `axis=0` because "students are axis 0" is exactly backwards, and
the only symptom is a wrong shape:

In [ ]:
scores = np.array([[80, 92, 70, 88],
                   [55, 64, 90, 71],
                   [98, 73, 85, 60]])        # 3 students x 4 quizzes

wrong = scores.mean(axis=0)      # BROKEN intent: shape (4,) = one value PER QUIZ
print("shape", wrong.shape, "- 4 values for 3 students? wrong axis")

fixed = scores.mean(axis=1)      # FIX: quiz axis disappears -> one value per student
print("shape", fixed.shape, "->", fixed.round(1))

Say the target out loud — "I want one number per *student*, so the quiz axis
must go" — instead of memorizing axis numbers.

### Checkpoint 9

1. `h = np.ones((5, 5))`. A classmate writes `h - h.sum(axis=1)` intending to
   subtract each row's total from that row. What does it actually compute, why
   does it run without error, and what is the fix?
2. For a `(12, 31)` array of one reading per day (rows = months, columns =
   days): which axis do you pass to get one average per month? Predict the
   result's shape.

## Checkpoint answers

### Checkpoint 1 answers

In [ ]:
radii = np.array([1.0, 2.0, 3.5])
print(np.pi * radii ** 2)                            # 1.
print(np.array([2, 4, 6]) ** np.array([1, 2, 3]))    # 2. [2, 16, 216] — paired powers

### Checkpoint 2 answers

In [ ]:
a = np.arange(15)
print(np.where(a > 10, 10, a))                # 1.

# 2. np.maximum(a, b) compares two arrays elementwise and returns an array of
#    the larger value from each pair; a.max() collapses one array to its single
#    largest element.

### Checkpoint 3 answers

In [ ]:
# 1. (5,1)+(1,6) -> (5, 6); (2,3)+(3,) -> (2, 3); (3,)+(4,) -> ERROR
print((np.ones((5, 1)) + np.ones((1, 6))).shape)
print((np.ones((2, 3)) + np.ones(3)).shape)
try:
    np.ones(3) + np.ones(4)
except ValueError as err:
    print("ERROR:", err)

# 2. Shapes are compared from the RIGHT: (4,) pads to (1, 4), so its 4 lines up
#    against the 5, not against the other 4. Position matters, not presence.

### Checkpoint 4 answers

In [ ]:
n = np.arange(1, 11)
print(n[:, None] * n[None, :])                # 1.

x = np.arange(4)
table = x[:, None] - x[None, :]
print(table.shape, table[2, 1])               # 2. (4, 4); table[2,1] = x[2]-x[1] = 1

### Checkpoint 5 answers

In [ ]:
values = np.array([3.0, -1.0, 4.0, -1.0, 5.0, -9.0, 2.0])

total = np.where(values < 0, 0, values).sum()        # 1.
print(total)

nums = np.array([4, 7, 10, 15, 22, 9])               # 2. a mask's .sum() counts Trues
print((nums % 2 == 0).sum())

### Checkpoint 6 answers

In [ ]:
def clip_total(values, hi):                  # 1.
    return np.minimum(values, hi).sum()      #    (np.where(values > hi, hi, values) also works)

print(clip_total(np.array([1.0, 8.0, 3.0]), 5.0))    # 1 + 5 + 3 = 9

# 2. Check it mechanically: search your own code for "for" and "while" (and
#    rerun it top to bottom) before submitting — the graders' check is literal.

### Checkpoint 7 answers

In [ ]:
s = np.arange(12).reshape(3, 4)
print(s.max(axis=0), "shape", s.max(axis=0).shape)   # 1. (4,): axis 0 disappears
print(s.sum(axis=1), "shape", s.sum(axis=1).shape)   #    (3,): axis 1 disappears
print(s.sum(axis=1).argmax())                        # 2. row index 2

### Checkpoint 8 answers

In [ ]:
s = np.arange(12).reshape(3, 4)
print(s / s.sum(axis=1, keepdims=True))              # 1. (3,4) / (3,1) broadcasts

# 2. s.sum(axis=1) has shape (3,), and (3, 4) vs (3,) compares 4 against 3 from
#    the right: incompatible. s.sum(axis=0) has shape (4,), and (3, 4) vs (4,)
#    lines the 4s up, so it broadcasts fine.

### Checkpoint 9 answers

In [ ]:
# 1. h.sum(axis=1) is (5,); against (5, 5) it broadcasts across ROWS, so each
#    COLUMN j has the j-th row-total subtracted — the wrong direction. It runs
#    because the grid is square (5 lines up with 5). Fix: keepdims=True.
h = np.ones((5, 5))
print((h - h.sum(axis=1, keepdims=True))[0])

# 2. One average per month = the day axis (axis 1) disappears -> shape (12,).
days = np.ones((12, 31))
print(days.mean(axis=1).shape)